In [1]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [2]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

display(run(load_queries("sql/01_scope.sql"), "scope_reconcile"))

,raw_rows,scoped_rows,rows_removed_by_scope
0,20405666,20405666,0


In [3]:
audit = load_queries("sql/01_audit_timestamps.sql")

for name in audit:
    print(f"\n=== {name} ===")
    display(run(audit, name))


=== wait_sign_summary ===


,wait_sign,n_rows,pct_rows
0,1_negative,199788,0.979
1,2_zero,208,0.001
2,3_positive,20205670,99.020



=== negative_magnitude ===


,gap_before_request,n_rows,min_wait_s,median_wait_s,max_wait_s
0,a: 60 min or more,4,-11148,-4470.5,-3705
1,b: 10 to 60 min,65649,-3175,-861.0,-600
2,c: 1 to 10 min,98114,-599,-276.0,-60
3,d: under 1 min,36021,-59,-1.0,-1



=== negative_by_company ===


,hvfhs_license_num,n_rows,n_negative,pct_negative,n_zero,pct_zero
0,HV0003,15356455,120967,0.788,169,0.001
1,HV0005,5049211,78821,1.561,39,0.001


... (6 outputs left out)


=== onscene_position_negative ===


,hvfhs_license_num,on_scene_position,n_rows
0,HV0003,1_on_scene_at_or_before_pickup,111313
1,HV0003,2_on_scene_between_pickup_and_request,9387
2,HV0003,3_on_scene_after_request,267
3,HV0005,0_on_scene_null,73567
4,HV0005,1_on_scene_at_or_before_pickup,5254



=== lyft_onscene_profile ===


,wav_request_flag,wav_match_flag,shared_request_flag,n_rows,n_with_on_scene
0,Y,Y,N,14076,14076
1,N,N,N,4385578,0
2,N,Y,N,649557,0


In [4]:
root = load_queries("sql/01_audit_negative_wait_root.sql")
for name in root:
    print(f"\n=== {name} ===")
    display(run(root, name))


=== negative_by_company_flags ===


,hvfhs_license_num,wav_request_flag,wav_match_flag,shared_request_flag,n_rows,n_negative,pct_negative
0,HV0003,N,N,N,13393994,105535,0.788
1,HV0003,N,Y,N,1088115,6062,0.557
2,HV0003,N,N,Y,771581,8017,1.039
3,HV0003,N,Y,Y,65691,674,1.026
4,HV0003,Y,Y,N,37074,679,1.831
5,HV0005,N,N,N,4385578,62663,1.429
6,HV0005,N,Y,N,649557,10904,1.679
7,HV0005,Y,Y,N,14076,5254,37.326



=== uber_order_patterns ===


,order_pattern,n_rows,pct_of_uber
0,A_normal: request<=on_scene<=pickup,15176848,98.830
1,B_wait_ok_but_on_scene_before_request,58321,0.380
2,C_negative: on_scene<=pickup<request,111313,0.725
3,D_negative: pickup<on_scene<=request,9387,0.061
4,E_negative: pickup<request<on_scene,267,0.002
5,F_wait_ok_but_on_scene_after_pickup,319,0.002



=== request_lateness_uber ===


,wait_group,request_after_on_scene,n_rows,median_wait_s
0,negative_wait,a: under 1 min,665,-19.0
1,negative_wait,b: 1 to 10 min,45712,-228.0
2,negative_wait,c: 10 to 60 min,74323,-758.0
3,nonneg_wait,a: under 1 min,2871,106.0
4,nonneg_wait,b: 1 to 10 min,30276,181.0
5,nonneg_wait,c: 10 to 60 min,25155,206.0
6,nonneg_wait,d: 60 min or more,19,1881.0


... (6 outputs left out)


=== negative_vs_normal_profile ===


,hvfhs_license_num,wait_group,n_rows,median_miles,median_trip_min,median_base_fare,median_driver_pay
0,HV0003,negative_wait,120967,9.190,27.716667,40.58,32.83
1,HV0003,nonneg_wait,15235488,2.860,15.016667,18.12,14.08
2,HV0005,negative_wait,78821,5.814,23.700000,29.38,25.07
3,HV0005,nonneg_wait,4970390,2.673,14.850000,17.38,13.21



=== exclusion_sensitivity ===


,hvfhs_license_num,n_rows,n_excluded,median_min_excluded,median_min_if_all_low,median_min_if_all_high,p90_min_excluded,p90_min_if_all_low,p90_min_if_all_high
0,HV0003,15356455,120967,3.800,3.783,3.817,8.033,8.017,8.267
1,HV0005,5049211,78821,4.083,4.033,4.117,8.033,8.000,8.433
